In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
import pickle

In [2]:
df = pd.read_csv("../dataset/processed_data.csv")

In [3]:
df

,Patient_Age,Patient_Gender,Symptom_Severity,Duration_of_Symptoms_Days,Pain_Score,Recommended_Department,Patient_Text
0,33,Female,High,10,9,Primary Care,Chief Complaint: Needs to see a doctor and age...
1,27,Male,Low,21,6,Gastroenterology,Chief Complaint: Caller reports abdominal pain...
2,38,Male,Medium,14,3,Scheduling,Chief Complaint: Caller reports needs speciali...
3,7,Female,Low,93,1,Dermatology,Chief Complaint: Patient is dealing with skin ...
4,17,Male,Medium,83,3,Pediatrics,Chief Complaint: Parent says child has fatigue...
...,...,...,...,...,...,...,...
9995,40,Female,Low,8,6,Nurse Triage,Chief Complaint: Concern about wants nurse gui...
9996,33,Male,High,10,7,Gastroenterology,"Chief Complaint: Caller reports cough, vomitin..."
9997,40,Male,Critical,0,7,Emergency Department,Chief Complaint: Possible life-threatening iss...
9998,37,Male,Low,58,5,Gastroenterology,Chief Complaint: Caller reports abdominal pain...


In [4]:
df["Patient_Text"] = (
    df["Patient_Text"]
        .str.lower()
        .str.replace(r'[^\w\s]', '', regex=True)
        .str.replace(r'\s+',' ', regex=True)
        .str.strip()
)

In [5]:
df["Patient_Text"]

0       chief complaint needs to see a doctor and agen...
1       chief complaint caller reports abdominal pain ...
2       chief complaint caller reports needs specialis...
3       chief complaint patient is dealing with skin l...
4       chief complaint parent says child has fatigue ...
                              ...                        
9995    chief complaint concern about wants nurse guid...
9996    chief complaint caller reports cough vomiting ...
9997    chief complaint possible lifethreatening issue...
9998    chief complaint caller reports abdominal pain ...
9999    chief complaint patient is dealing with blood ...
Name: Patient_Text, Length: 10000, dtype: object

In [6]:
df["Patient_Gender"].value_counts()

Patient_Gender
Female    4956
Male      4770
Other      274
Name: count, dtype: int64

In [7]:
df["Symptom_Severity"].value_counts()

Symptom_Severity
Medium      3895
Low         2703
High        2523
Critical     879
Name: count, dtype: int64

In [8]:
severity_mapping = {
    "Low" : 0,
    "Medium" : 1,
    "High" : 2,
    "Critical" : 3
}

In [9]:
df["Symptom_Severity"] = df["Symptom_Severity"].map(severity_mapping)

In [10]:
df["Symptom_Severity"].value_counts()

Symptom_Severity
1    3895
0    2703
2    2523
3     879
Name: count, dtype: int64

In [11]:
gender_encoded = pd.get_dummies(df["Patient_Gender"], dtype=int)

In [12]:
df = pd.concat([df, gender_encoded], axis=1)

In [13]:
df = df.drop(columns=["Patient_Gender"])

In [14]:
df["Recommended_Department"].nunique()

17

In [15]:
new_df = sorted(df["Recommended_Department"].unique())

In [16]:
departments_mapping = {
    'Billing' : 0,
    'Cardiology' : 1,
    'Dermatology' : 2,
    'Emergency Department' : 3,
    'Gastroenterology' : 4,
    'Mental Health' : 5,
    'Neurology' : 6,
    'Nurse Triage' : 7,
    'OB-GYN' : 8,
    'Oncology' : 9,
    'Orthopedics' : 10,
    'Pediatrics' : 11,
    'Pharmacy' : 12,
    'Primary Care' : 13,
    'Pulmonology' : 14,
    'Scheduling' : 15,
    'Urgent Care' : 16
}

In [17]:
df["Recommended_Department"] = df["Recommended_Department"].map(departments_mapping)

In [18]:
df["Recommended_Department"]

0       13
1        4
2       15
3        2
4       11
        ..
9995     7
9996     4
9997     3
9998     4
9999    13
Name: Recommended_Department, Length: 10000, dtype: int64

In [19]:
# TF-IDF Vectorization

In [20]:
tfidf = TfidfVectorizer()

In [21]:
X_text = tfidf.fit_transform(df["Patient_Text"])

In [22]:
type(X_text)

scipy.sparse._csr.csr_matrix

In [23]:
X_text.shape

(10000, 1128)

In [24]:
X_structured = df[['Patient_Age', 'Symptom_Severity',
       'Duration_of_Symptoms_Days', 'Female', 'Male', 'Other']]

In [25]:
X_structured.shape

(10000, 6)

In [26]:
type(X_structured)

pandas.core.frame.DataFrame

In [27]:
X_structured_sparse = csr_matrix(X_structured)

In [28]:
type(X_structured_sparse)

scipy.sparse._csr.csr_matrix

In [29]:
X_structured_sparse.shape

(10000, 6)

In [30]:
X = hstack([X_text, X_structured_sparse])

In [31]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 146852 stored elements and shape (10000, 1134)>

In [32]:
y = df["Recommended_Department"]

In [33]:
X.shape

(10000, 1134)

In [34]:
y.shape

(10000,)

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
                                        X,
                                        y,
                                        test_size=0.2,
                                        random_state = 42
                                    )

In [36]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(8000, 1134)
(2000, 1134)
(8000,)
(2000,)


In [37]:
model = LogisticRegression(max_iter=2000, random_state=42)

In [38]:
model.fit(X_train, y_train)

C:\Users\Dell\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [39]:
y_pred = model.predict(X_test)

In [40]:
accuracy = accuracy_score(y_test, y_pred)

In [41]:
print("Accuracy:", accuracy)

Accuracy: 0.844


In [42]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99        75
           1       0.78      0.63      0.70        95
           2       0.85      0.97      0.91        76
           3       0.73      0.64      0.68       304
           4       0.86      0.93      0.89       135
           5       0.97      0.97      0.97        79
           6       0.88      0.96      0.92        73
           7       0.84      0.89      0.86       157
           8       0.90      0.92      0.91        71
           9       0.86      0.98      0.92        57
          10       0.88      0.95      0.91       126
          11       0.75      0.68      0.71       139
          12       0.98      0.98      0.98        64
          13       0.85      0.91      0.88       233
          14       0.82      0.78      0.80       106
          15       0.97      0.90      0.94        72
          16       0.82      0.82      0.82       138

    accuracy              

In [43]:
from sklearn.tree import DecisionTreeClassifier

In [44]:
dt_model = DecisionTreeClassifier(random_state=42)

In [45]:
dt_model.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [46]:
y_pred_dt = dt_model.predict(X_test)

In [47]:
accuracy_dt = accuracy_score(y_test, y_pred_dt)

In [48]:
accuracy_dt

0.79

In [49]:
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print("Decision Tree Accuracy:", accuracy_dt)

Decision Tree Accuracy: 0.79


In [50]:
print(classification_report(y_test, y_pred_dt))

              precision    recall  f1-score   support

           0       0.95      0.96      0.95        75
           1       0.55      0.57      0.56        95
           2       0.88      0.87      0.87        76
           3       0.62      0.60      0.61       304
           4       0.88      0.88      0.88       135
           5       0.97      0.96      0.97        79
           6       0.78      0.81      0.79        73
           7       0.84      0.79      0.81       157
           8       0.87      0.83      0.85        71
           9       0.86      0.88      0.87        57
          10       0.90      0.89      0.90       126
          11       0.69      0.81      0.74       139
          12       0.97      0.95      0.96        64
          13       0.81      0.85      0.83       233
          14       0.78      0.72      0.75       106
          15       0.89      0.86      0.87        72
          16       0.74      0.71      0.73       138

    accuracy              

In [51]:
# Muiltinomial Naive Bayes

In [52]:
from sklearn.naive_bayes import MultinomialNB

In [53]:
nb_model = MultinomialNB()

In [54]:
nb_model.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [55]:
y_pred_nb = nb_model.predict(X_test)

In [56]:
accuracy_nb = accuracy_score(y_test, y_pred_nb)

In [57]:
accuracy_nb

0.5645

In [58]:
print(classification_report(y_test, y_pred_nb))

              precision    recall  f1-score   support

           0       0.97      0.97      0.97        75
           1       0.51      0.24      0.33        95
           2       0.81      0.67      0.73        76
           3       0.36      0.84      0.50       304
           4       0.64      0.48      0.55       135
           5       0.96      0.63      0.76        79
           6       0.90      0.38      0.54        73
           7       0.62      0.50      0.55       157
           8       0.82      0.38      0.52        71
           9       0.58      0.19      0.29        57
          10       0.83      0.41      0.55       126
          11       0.42      0.52      0.47       139
          12       0.98      0.86      0.92        64
          13       0.51      0.65      0.57       233
          14       0.90      0.26      0.41       106
          15       1.00      0.78      0.88        72
          16       0.69      0.39      0.50       138

    accuracy              

In [59]:
from sklearn.ensemble import RandomForestClassifier

In [60]:
rf_model = RandomForestClassifier(random_state=42)

In [61]:
rf_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [62]:
y_pred_rf = rf_model.predict(X_test)

In [63]:
accuracy_rf = accuracy_score(y_test, y_pred_rf)

In [64]:
accuracy_rf

0.8445

In [65]:
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99        75
           1       0.69      0.62      0.65        95
           2       0.87      0.99      0.93        76
           3       0.80      0.60      0.69       304
           4       0.85      0.93      0.89       135
           5       0.97      0.97      0.97        79
           6       0.82      0.95      0.88        73
           7       0.82      0.91      0.86       157
           8       0.85      0.99      0.92        71
           9       0.87      0.96      0.92        57
          10       0.88      1.00      0.94       126
          11       0.77      0.63      0.69       139
          12       0.97      1.00      0.98        64
          13       0.84      0.91      0.87       233
          14       0.80      0.86      0.83       106
          15       0.97      0.90      0.94        72
          16       0.81      0.82      0.81       138

    accuracy              

In [66]:
with open("../model.pkl", "wb") as file:
    pickle.dump(rf_model, file)

with open("../vectorizer.pkl", "wb") as file:
    pickle.dump(tfidf, file)

print("model and vectorizer files saved successfully!")

model and vectorizer files saved successfully!


In [67]:
with open("../model.pkl", "rb") as file:
    saved_model = pickle.load(file)

print(type(saved_model))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [68]:
reverse_department_mapping = {
    value: key
    for key, value in departments_mapping.items()
}
print(reverse_department_mapping)

{0: 'Billing', 1: 'Cardiology', 2: 'Dermatology', 3: 'Emergency Department', 4: 'Gastroenterology', 5: 'Mental Health', 6: 'Neurology', 7: 'Nurse Triage', 8: 'OB-GYN', 9: 'Oncology', 10: 'Orthopedics', 11: 'Pediatrics', 12: 'Pharmacy', 13: 'Primary Care', 14: 'Pulmonology', 15: 'Scheduling', 16: 'Urgent Care'}


In [69]:
with open("../department_mapping.pkl", "wb") as file:
    pickle.dump(reverse_department_mapping, file)

print("reverse_department_mapping is created successfully")

reverse_department_mapping is created successfully


In [70]:
with open("../department_mapping.pkl", "wb") as file:
    pickle.dump(reverse_department_mapping, file)

print("department_mapping file saved succesfully")

department_mapping file saved succesfully


In [1]:
import pickle
import re
from scipy.sparse import csr_matrix, hstack

with open("../model.pkl", "rb") as file:
    model = pickle.load(file)

with open("../vectorizer.pkl", "rb") as file:
    tfidf = pickle.load(file)

with open("../department_mapping.pkl", "rb") as file:
    department_mapping = pickle.load(file)

print("Files loaded successfully")

Files loaded successfully


In [37]:
complaint = "chest pain"
age = 23
gender = "Male"
duration = 3
severity = "Medium"

In [38]:
print("Complaint:", complaint)
print("Age:", age)
print("Gender:", gender)
print("Duration:", duration)
print("Severity:", severity)

Complaint: chest pain
Age: 23
Gender: Male
Duration: 3
Severity: Medium


In [39]:
complaint = re.sub(
    r"[^\w\s]",
    "",
    complaint.lower()
).strip()

print("Cleaned complaint:", complaint)

Cleaned complaint: chest pain


In [40]:
X_text = tfidf.transform([complaint])

print("TF-IDF shape:", X_text.shape)

TF-IDF shape: (1, 1128)


In [41]:
female = 0
male = 0
other = 0

if gender.lower() == "male":
    male = 1
elif gender.lower() == "female":
    female = 1
else:
    other = 1

print("Female:", female)
print("Male:", male)
print("Other:", other)

Female: 0
Male: 1
Other: 0


In [42]:
severity_mapping = {
    "low": 0,
    "medium": 1,
    "high": 2,
    "critical": 3
}

severity_value = severity_mapping[severity.lower()]

print("Severity value:", severity_value)

Severity value: 1


In [43]:
X_structured = [[
    int(age),
    severity_value,
    int(duration),
    female,
    male,
    other
]]

print("Structured features:", X_structured)

Structured features: [[23, 1, 3, 0, 1, 0]]


In [44]:
X_structured = csr_matrix(X_structured)

print("Structured shape:", X_structured.shape)

Structured shape: (1, 6)


In [45]:
X_input = hstack([
    X_text,
    X_structured
])

print("X_input shape:", X_input.shape)

X_input shape: (1, 1134)


In [46]:
prediction = model.predict(X_input)[0]
department = department_mapping[prediction]

In [47]:
department = department_mapping[prediction]

print("Department:", department)

Department: Cardiology
